# Feature Map 크기 계산 및 실습 


$$
H_{out}=\left\lfloor \frac{H_{in}+2P_h-D_h,(K_h-1)-1}{S_h}+1 \right\rfloor
$$

$$
W_{out}=\left\lfloor \frac{W_{in}+2P_w-D_w,(K_w-1)-1}{S_w}+1 \right\rfloor
$$


$ (H_{in}, W_{in}) $: 입력 크기  
$ (K_h, K_w) $: 커널 크기  
$ (S_h, S_w) $: 스트라이드  
$ (P_h, P_w) $: 패딩  
$ (D_h, D_w) $: dilation(기본 1)  


In [27]:
import torch

# 원본 데이터: (H, W, C) 형태로 입력
raw_input_tensor = torch.tensor(
    [
        [[1, 0, 1],
         [2, 1, 0],
         [0, 1, 2],
         [2, 0, 0],
         [1, 1, 1]],

        [[0, 0, 0],
         [1, 2, 0],
         [1, 1, 0],
         [0, 1, 1],
         [0, 0, 1]],

        [[1, 0, 1],
         [0, 0, 0],
         [2, 2, 2],
         [0, 0, 1],
         [1, 1, 2]],

        [[0, 0, 1],
         [1, 0, 0],
         [1, 1, 1],
         [2, 1, 0],
         [0, 0, 0]],

        [[1, 1, 0],
         [0, 0, 0],
         [1, 2, 1],
         [0, 2, 2],
         [0, 0, 0]]
    ],
    dtype=torch.float32
)

# 원본 필터 데이터: (kH, kW, C)
raw_filter_tensor = torch.tensor(
    [
        [[1, 1, 2],
         [0, 2, 0]],

        [[0, 0, 0],
         [1, 1, 0]]
    ],
    dtype=torch.float32
)

# Conv2d 입력 형태 (N, C, H, W)로 변환
input_tensor = raw_input_tensor.permute(2, 0, 1).unsqueeze(0)

# Conv2d 가중치 형태 (out_channels, in_channels, kH, kW)로 변환
filter_tensor = raw_filter_tensor.permute(2, 0, 1).unsqueeze(0)

print("raw_input_tensor shape (H, W, C):", raw_input_tensor.shape)
print("input_tensor shape (N, C, H, W):", input_tensor.shape)
print("raw_filter_tensor shape (kH, kW, C):", raw_filter_tensor.shape)
print("filter_tensor shape (out, in, kH, kW):", filter_tensor.shape)

raw_input_tensor shape (H, W, C): torch.Size([5, 5, 3])
input_tensor shape (N, C, H, W): torch.Size([1, 3, 5, 5])
raw_filter_tensor shape (kH, kW, C): torch.Size([2, 2, 3])
filter_tensor shape (out, in, kH, kW): torch.Size([1, 3, 2, 2])


In [28]:
import torch.nn as nn

conv_layer = nn.Conv2d(in_channels=3, out_channels=1, kernel_size=2, bias=True)

with torch.no_grad():
    conv_layer.weight.copy_(filter_tensor)
    conv_layer.bias.zero_()

output_tensor = conv_layer(input_tensor)

print("Output shape after convolution:", output_tensor.shape)
print("Output tensor after convolution:\n", output_tensor)

Output shape after convolution: torch.Size([1, 1, 4, 4])
Output tensor after convolution:
 tensor([[[[ 8.,  7.,  6.,  4.],
          [ 4.,  9.,  4.,  5.],
          [ 4.,  6., 11.,  4.],
          [ 2.,  6.,  8.,  3.]]]], grad_fn=<ConvolutionBackward0>)
